# **Analisis Sentimen Publik terhadap AI yang Menggantikan Pekerjaan Manusia**

**Mata Kuliah:** Data Mining

**Nama Anggota Kelompok:**

1.   Muhammad Rafa Enrico (Ketua)
2.   Roihan Saputra

**Kelas TI-3C**

**Program Studi:** D4 Teknologi Rekayasa Komputer, Politeknik Negeri Semarang

## Rumusan Masalah
1. Bagaimana sebaran sentimen publik (positif, negatif, netral) terhadap isu AI yang menggantikan pekerjaan manusia berdasarkan komentar YouTube?
2. Algoritma klasifikasi mana (Naive Bayes, SVM, atau Random Forest) yang memberikan performa terbaik untuk diimplementasikan sebagai model automasi deteksi sentimen publik?

## Tujuan
1. Mengetahui sebaran sentimen publik terhadap isu AI yang menggantikan pekerjaan manusia.
2. Menemukan model klasifikasi yang paling tangguh (robust) untuk mengotomatisasi proses pelabelan sentimen pada data komentar di masa depan.

# **1. Import Library**

In [3]:
!pip install youtube-comment-downloader langdetect transformers torch --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from langdetect import detect, DetectorFactory, LangDetectException
from transformers import pipeline
import torch
import joblib
import re

# **2. Load Datasets and Data Understanding**

**Feature yang Kami Ambil:**
1. `video_url`: Link atau URL dari 6 Video yang kami ambil dari youtube
2. `author` : pembuat/penulis comment dari 6 video youtube yang berbeda beda
3. `comment` : hasil comment yang di hasilkan oleh para author
4. `likes` : jumlah yang menyukai comment youtube yang di keluarkan author
5. `time` : waktu rilis comment disaat tanggal melakukan Web Scrabbing


In [4]:
df = pd.read_csv("/content/youtube_comments.csv")
df

,video_url,author,comment,likes,time
0,https://www.youtube.com/watch?v=t2WmNcToB3I,@aprie_rdw,Bahas imigrant maroko yg nyebrang ke spanyol d...,413,2 maanden geleden
1,https://www.youtube.com/watch?v=t2WmNcToB3I,@robidarmawan71198,Menurut saya jika semua pengusaha berpikiran s...,189,2 maanden geleden
2,https://www.youtube.com/watch?v=t2WmNcToB3I,@wooshstar,jujur ya sebagai seorang yang bekerja di bidan...,6,2 maanden geleden (bewerkt)
3,https://www.youtube.com/watch?v=t2WmNcToB3I,@kadaladal7,Gw sebagai illustrator susah bener dapet klien...,588,2 maanden geleden
4,https://www.youtube.com/watch?v=t2WmNcToB3I,@Bagas_betok,Niat untung -> efesiensi phk->produksi mening...,85,2 maanden geleden
...,...,...,...,...,...
4453,https://www.youtube.com/watch?v=0LOHWGQwKtM,@HMF-7s,There’s a real awakening that cannot be stoppe...,0,5 maanden geleden
4454,https://www.youtube.com/watch?v=0LOHWGQwKtM,@fr0z3nk0,I'm a software engineer with 20 years of exper...,0,5 maanden geleden (bewerkt)
4455,https://www.youtube.com/watch?v=0LOHWGQwKtM,@nalen5050,so many way to discard that... I'll just take ...,2,5 maanden geleden
4456,https://www.youtube.com/watch?v=0LOHWGQwKtM,@georgioszampoukis,@nalen5050 Are you a software engineer? Beca...,0,5 maanden geleden


**Hasil dari function info :**

seperti yang bisa kita lihat bahwa semua tipe data
atau data types dari
hasil scrabbing adalah berbentuk Object semua/string/kategorikal.
dan jumlah baris data yang kami miliki adalah 4458

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4458 entries, 0 to 4457
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   video_url  4458 non-null   object
 1   author     4458 non-null   object
 2   comment    4458 non-null   object
 3   likes      4458 non-null   object
 4   time       4458 non-null   object
dtypes: object(5)
memory usage: 174.3+ KB


**Hasil dari Describe :**

1. count (Jumlah data) : Total baris data yang terisi (tidak kosong/null) pada kolom tersebut.
2. Unique (Keunikan / Variasi Data): Jumlah nilai yang benar-benar berbeda (tidak kembar) di dalam kolom tersebut.
3. Top (Nilai Paling Sering Muncul / Modus) : Data atau teks apa yang frekuensi kemunculannya paling tinggi di kolom itu.
4. Freq (Frequency / Frekuensi) :
Seberapa banyak si nilai top tadi muncul di dalam dataset.


Tabel ini menunjukkan karakteristik data teks kita. Count adalah total data yang kita punya, Unique menunjukkan seberapa banyak variasi datanya, Top adalah isi data yang paling dominan atau sering muncul, dan Freq menunjukkan berapa kali nilai Top tersebut muncul di dalam dataset

In [6]:
df.describe().T

,count,unique,top,freq
video_url,4458,6,https://www.youtube.com/watch?v=7Pq-S557XQU,1500
author,4458,3320,@edward-broo,36
comment,4458,4406,Up,6
likes,4458,241,0,2400
time,4458,62,2 maanden geleden,933


# **2. Preprocessing Data**

Pada tahap preprocessing, kami menerapkan pendekatan yang sistematis dan terstruktur untuk memastikan kualitas data mentah sebelum masuk ke tahap analisis dan pemodelan. Setiap langkah pembersihan dilakukan dengan konsisten menggunakan pola 3 tahap (3-Step Framework):

1. Cek: Menghitung jumlah data yang bermasalah dan menampilkan contohnya.

2. Bersihkan: Menghapus baris atau memperbaiki teksnya.

3. Verifikasi: Memastikan masalah pada tahap tersebut sudah selesai.

**Daftar Langkah Pembersihan:**

1. Nilai kosong: Hapus baris yang memiliki nilai kosong.

2. Duplikat baris persis: Hapus baris yang identik di seluruh kolom dan sisakan satu.

3. Spam judi online: Hapus baris komentar yang terindikasi promosi judi online.

4. Karakter tak terlihat (zero-width space): Hapus karakter tersembunyi atau Unicode non-standar dari teks.

5. URL di dalam komentar: Hapus tautan atau link web di dalam isi komentar.

6. Mention (@username): Hapus pemanggilan akun lain di dalam teks komentar.

7. Baris baru dan spasi berlebih: Ganti karakter newline dan spasi ganda menjadi satu spasi.

8. Duplikat teks komentar: Hapus komentar dengan isi teks yang sama persis dan sisakan satu.

9. Komentar kosong atau terlalu pendek: Hapus baris komentar yang terlalu singkat atau tidak berkonten.

10. Komentar tanpa huruf atau angka: Hapus baris komentar yang hanya berisi simbol atau emotikon saja.

11. Format kolom likes: Ubah tipe data jumlah like dari teks menjadi angka.

12. Format kolom waktu: Ubah format waktu menjadi umur komentar (hari), perkiraan tanggal, dan status diedit.

13. Deteksi bahasa: Tambah kolom bahasa dan hapus komentar dengan bahasa yang tidak terdeteksi.

14. Ringkasan dan verifikasi akhir: Lakukan audit akhir pada seluruh DataFrame.

15. Menyimpan hasil: Simpan data hasil pembersihan ke file CSV baru.

## **2.1 Nilai kosong atau Null**

**Kenapa Data NULL (Nilai Kosong) Harus Dihapus/Ditangani?**

1. **Bikin Kode Crash / Error:** Sebagian besar algoritma Machine Learning (seperti SVM, Naive Bayes, Logistic Regression, hingga Neural Network) menggunakan perhitungan matematika dasar (perkalian matriks dan jarak vector). Algoritma ini tidak bisa mengolah nilai kosong (NaN/Null), sehingga program akan langsung melempar error saat training.

2. **Mengurangi Kualitas Informasi:**
 Dalam kasus analisis sentimen, jika kolom teks komentar berisi Null, artinya tidak ada informasi emosi/opini sama sekali yang bisa dipelajari oleh model.

3. **Memicu Bias Saat Feature Extraction:**
 Ketika kamu melakukan ekstraksi fitur menggunakan TF-IDF, matriks representasi kata yang terbentuk bisa menjadi tidak konsisten atau rusak jika ada baris tanpa konten teks.

dari hasil yang kita dapatkan untuk melihat data yang null atau not available kita mendapatkan 0

In [7]:
null_counts = df.isnull().sum()
df_null = pd.DataFrame({
    "Nama Kolom": null_counts.index,
    "jumlah null": null_counts.values})
display(df_null)

total_null = null_counts.sum()
if total_null == 0:
  print("gaada null bang")
else:
  print(f"ada null nih bang sebanyak {null_counts}")

,Nama Kolom,jumlah null
0,video_url,0
1,author,0
2,comment,0
3,likes,0
4,time,0


gaada null bang


## **2.2 Nilai Data yang Duplikat**

**Kenapa Data DUPLIKAT Harus Dihapus?**

1. **Mencegah Data Leakage (Kebocoran Data):** Ini alasan paling krusial. Saat kamu melakukan splitting data menjadi Data Train dan Data Test (misal rasio 80:20), jika ada data duplikat, kalimat yang sama persis bisa masuk ke Data Train sekaligus ke Data Test. Akibatnya, model dites menggunakan data yang sudah pernah dia "lihat" saat latihan.

2. **Akurasi Palsu (Overfitting / False High Accuracy):** Karena kebocoran data di atas, nilai akurasi modelmu bisa kelihatan sangat tinggi (misal 98%), padahal itu akurasi semu/palsu. Model sebenarnya hanya "menghafal" komentar duplikat tersebut, bukan benar-benar pintar memahami pola kalimat baru.

3. **Penyimpangan Bobot (Bias Model)**: Jika satu komentar tertentu (misal komentar spam/bot yang sama) muncul 10–20 kali, algoritma akan menganggap kata-kata di dalam komentar itu sangat penting secara berlebihan. Ini merusak pembobotan kata pada TF-IDF dan membuat model condong/bias ke jenis komentar duplikat tersebut.

4. **Pemborosan Memori & Waktu Komputasi:** Memproses ribuan baris teks yang sama persis hanya membuang-buang RAM dan memperlambat waktu training tanpa memberikan nilai tambah bagi pengetahuan model.


dan seperti yang kita lihat, kami mendapatkan 11 data duplikat yang ada di dalam datasets kami

In [8]:
dup_counts = df.duplicated().sum()
df_dup = pd.DataFrame({
    "Nama Kolom": ["duplicated"],
    "jumlah_dup": [dup_counts]
})

display(df_dup)

if dup_counts == 0:
  print("aman gaada dup bang")
else:
  print(f"aish ada dup nih bang sebanyak {dup_counts}")

,Nama Kolom,jumlah_dup
0,duplicated,11


aish ada dup nih bang sebanyak 11


In [9]:
total_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat ditemukan: {total_duplikat}")

df = df.drop_duplicates().reset_index(drop=True)

print(f"✅ Berhasil menghapus {total_duplikat} baris data duplikat.")
print(f"📊 Jumlah baris dataset sekarang: {len(df)}")

Jumlah data duplikat ditemukan: 11
✅ Berhasil menghapus 11 baris data duplikat.
📊 Jumlah baris dataset sekarang: 4447


## **2.3. Data Spam judi online**

Komentar spam judi online wajib dihapus dari dataset karena melanggar prinsip dasar Machine Learning: Garbage In, Garbage Out (data sampah akan menghasilkan prediksi sampah).

Berikut adalah dampak fatal jika data tersebut tidak dibuang dari proyek klasifikasimu:

Tidak Relevan dengan Topik (Out of Topic): Fokus penelitianmu adalah sentimen publik terhadap isu AI yang menggantikan pekerjaan manusia. Komentar promosi seperti "slot", "gacor", atau "maxwin" sama sekali tidak mengandung opini positif, negatif, atau netral terkait isu AI.

Merusak Akurasi Model Klasifikasi: Model Naive Bayes, SVM, dan Random Forest belajar mengenali pola berdasarkan frekuensi kata (misalnya melalui TF-IDF). Jika spam dibiarkan, algoritma akan kebingungan dan malah merekam kosakata judi online sebagai "kata penting". Ini membuat performa dan akurasi model anjlok saat memprediksi komentar asli.

Memicu Bias pada Kesimpulan Sentimen: Iklan judi online sangat sering menggunakan kata-kata bernada ajakan atau euforia yang rentan diklasifikasikan sebagai sentimen "Positif" oleh mesin. Jika ratusan iklan slot masuk ke label Positif, maka persentase akhir sentimen publik terhadap AI akan menjadi cacat dan tidak merepresentasikan opini masyarakat yang sebenarnya.

In [10]:
# 1. Pola spam dari temanmu (Tanda (?i) dipindah ke paling depan agar TIDAK ERROR)
pola_spam = r"(?i)[\U0001D400-\U0001D7FF]|garuda\s*ho\s*ki"

# 2. Fungsi cek spam dari temanmu (tanpa bikin kolom baru)
def cek_spam(data):
    return data["comment"].astype(str).str.contains(pola_spam, regex=True)

# 3. Hitung jumlah spam awal
total_spam = cek_spam(df).sum()
print(f"Komentar terindikasi spam judi online: {total_spam}")

# 4. EKSEKUSI PENGHAPUSAN: Ambil data yang BUKAN spam (~), lalu reset index
df = df[~cek_spam(df)].reset_index(drop=True)

# 5. Verifikasi
print(f"✅ Berhasil menghapus {total_spam} komentar spam.")
print(f"📊 Sisa baris dataset sekarang: {len(df)}")

Komentar terindikasi spam judi online: 34
✅ Berhasil menghapus 34 komentar spam.
📊 Sisa baris dataset sekarang: 4413


## **2.4 Karakter Tak Terlihat (Zero-Width Space)**
Sebagian komentar balasan diawali karakter tak terlihat seperti `\u200b` (zero-width space). Karakter ini tidak tampak di layar dan tidak dianggap spasi oleh Python, sehingga bisa lolos dari proses perapian spasi dan ikut terbaca sebagai bagian dari kata.

Mulai langkah ini, perapian teks dilakukan di kolom baru `comment_clean`. Kolom `comment` asli tidak diubah.

Karakter yang dihapus: `\u200b` (zero-width space), `\u200c` (zero-width non-joiner), `\u2060` (word joiner), dan `\ufeff` (byte order mark).

Alasan utama kenapa zero-width space (karakter tak terlihat) haram hukumnya ada di dalam dataset teks (terutama untuk NLP dan Analisis Sentimen) adalah karena perbedaan cara pandang mata manusia dan cara baca mesin.

berikut poin-poin mengapa tidak boleh:

1. Membuat Mesin Gagal Mengenali Kata yang Sama (Tokenisasi Kacau)
Di mata kita, kata "Pekerjaan" dan "Pe<zero-width>kerjaan" terlihat 100% sama persis. Tapi bagi mesin (terutama saat kamu mengubah teks jadi angka pakai TF-IDF), itu dianggap sebagai dua kata yang sama sekali berbeda.

2. Merusak Perhitungan Bobot dan Frekuensi Kata
Algoritma seperti Naive Bayes sangat bergantung pada perhitungan "seberapa sering suatu kata muncul". Kalau ada 100 orang berkomentar kata "Bagus", tapi 30 di antaranya terselip karakter gaib, mesin hanya menghitung "Bagus" muncul 70 kali. Bobot kata sentimen yang penting jadi terpecah dan melemah.

3. Menurunkan Akurasi Model Klasifikasi
Karena kosakata (vocabulary) yang masuk ke dalam model menjadi kotor dan membengkak oleh variasi kata-kata "gaib", model algoritma (SVM, Naive Bayes, maupun Random Forest) akan kebingungan mencari pola yang valid. Ujung-ujungnya, performa akurasi, Precision, dan Recall modelmu akan anjlok.

seperti yang dapat kita lihat bahwa ada 273 comment yang memiliki Zero-widht space

In [11]:
# 1. Definisikan pola karakter tak terlihat
pola_tak_terlihat = r"[\u200b\u200c\u2060\ufeff]"


# 2. Fungsi pengecekan langsung pada kolom 'comment'
def cek_tak_terlihat(data):
    return data["comment"].astype(str).str.contains(pola_tak_terlihat, regex=True)


# 3. Hitung total komentar yang berdampak
print(f"Komentar yang mengandung karakter tak terlihat: {cek_tak_terlihat(df).sum()}")

# 4. Tampilkan contoh karakter tersembunyi
contoh_idx = df.index[cek_tak_terlihat(df)][0]
print("\nContoh (repr menampilkan karakter tersembunyi):")
print(repr(df.loc[contoh_idx, "comment"][:80]))

Komentar yang mengandung karakter tak terlihat: 273

Contoh (repr menampilkan karakter tersembunyi):
'\u200b\xa0@SepulangSekolah\xa0bahas juga soal perusahaan AI yang beli buku besar-besaran bu'


In [12]:
df["comment"] = df["comment"].astype(str).str.replace(pola_tak_terlihat, "", regex=True)

sisa_tak_terlihat = cek_tak_terlihat(df).sum()
print(f"✅ Karakter tak terlihat berhasil dibersihkan!")
print(f"Sisa komentar yang masih mengandung karakter gaib: {sisa_tak_terlihat}")

✅ Karakter tak terlihat berhasil dibersihkan!
Sisa komentar yang masih mengandung karakter gaib: 0


## **2.5 URL di Dalam Komentar**
Tautan (misalnya `https://...` atau `www...`) tidak bermakna untuk analisis sentimen. Namun, tidak semua komentar ber-URL adalah spam. Sebagian berisi opini yang menyertakan tautan sebagai sumber. Karena itu, yang dihapus **hanya tautannya**, bukan seluruh komentarnya.

Berikut adalah 3 alasan logis dan teknis mengapa URL harus dibersihkan:

1. Tidak Memiliki Nilai Sentimen (Zero Sentiment Value)
URL atau link (seperti https://... atau www...) hanyalah alamat navigasi web. Teks tersebut sama sekali tidak mengandung opini, emosi, atau argumen terkait isu AI yang menggantikan pekerjaan manusia. Menyimpan URL tidak akan membantu model dalam membedakan mana komentar Positif, Negatif, atau Netral.

2. Menjadi "Sampah" (Noise) yang Mengacaukan Perhitungan TF-IDF
Untuk algoritma yang kelompokmu pakai (Naive Bayes, SVM, dan Random Forest), mesin belajar mengenali pola dari frekuensi kata. Kalau URL dibiarkan, mesin akan menganggap setiap link yang berbeda sebagai "kosakata baru". Ini akan membuat kamus kata (vocabulary) datasetmu membengkak secara sia-sia oleh ratusan teks link aneh, yang ujung-ujungnya membuat model kebingungan dan menurunkan akurasi prediksinya.

3. Memboroskan Batas Token pada Model Transformer
Karena kamu juga bermain dengan Pre-trained Transformer, perlu diingat bahwa model canggih ini membaca teks dengan cara memecahnya menjadi token (potongan kata). Sebuah URL panjang bisa menghabiskan belasan hingga puluhan token hanya untuk membaca simbol seperti http, ://, www, .com, dll. Padahal, batas token per kalimat itu terbatas. Jika kuota token habis terpakai untuk membaca URL, opini asli dari netizen di belakangnya malah terpotong dan tidak sempat dibaca oleh mesin.

In [13]:
# 1. Definisikan pola untuk mendeteksi URL (http, https, atau www)
pola_url = r"https?://\S+|www\.\S+"

# 2. Fungsi untuk mengecek URL (ditambah astype(str) biar aman dari error data kosong)
def cek_url(data):
    return data["comment"].astype(str).str.contains(pola_url, regex=True)

# 3. Tampilkan jumlah komentar yang ada URL-nya
print(f"Komentar yang mengandung URL: {cek_url(df).sum()}")

# 4. Tampilkan 5 contoh komentarnya di dalam tabel rapi
print("\nContoh 5 komentar yang ada URL-nya:")
df_contoh = pd.DataFrame({
    "Komentar": df.loc[cek_url(df), "comment"].head(5).str[:150].str.replace("\n", " ")
})

# Gunakan fungsi bawaan Colab/Jupyter untuk menampilkan tabel
display(df_contoh)

Komentar yang mengandung URL: 11

Contoh 5 komentar yang ada URL-nya:


,Komentar
712,https://youtube.com/shorts/pjT0ubmENWk?si=rUss...
836,https://youtu.be/difFzihUhPo?si=kogGg2E2cUw8EqbV
1607,"Soal constraint AI adalah energi, ini elon mus..."
1638,Bukan teori saya. Ini jensen huang sama elon m...
1648,I was thinking this video was getting out of d...


In [14]:
# 1. EKSEKUSI PENGHAPUSAN: Ganti teks URL menjadi spasi kosong
df["comment"] = df["comment"].astype(str).str.replace(pola_url, " ", regex=True)

# 2. Rapikan spasi ganda/berlebihan akibat penghapusan kata
df["comment"] = df["comment"].str.replace(r"\s+", " ", regex=True).str.strip()

# 3. Print verifikasi hasil akhir (memanggil fungsi cek_url dari cell sebelumnya)
sisa_url = cek_url(df).sum()
print(f"✅ Semua teks URL berhasil dibersihkan dari dalam komentar!")
print(f"📊 Sisa komentar yang masih mengandung URL: {sisa_url}")

✅ Semua teks URL berhasil dibersihkan dari dalam komentar!
📊 Sisa komentar yang masih mengandung URL: 0


## **2.6 Mention (@username) dan newline (spasi berlebih)

untuk `@username` Sangat mirip dengan URL sifatnya

Secara konsep, menghapus `newline` (enter/\n) dan spasi ganda (Step 2.7) itu wajib karena dua alasan:

Mencegah Error saat Save ke CSV: Karakter enter (\n) di dalam komentar sering bikin baris data berantakan atau "patah" saat nanti kamu simpan (export) datasetnya ke Excel/CSV baru.

Kerapian Tokenisasi: Memastikan teks benar-benar rapi hanya dengan satu spasi antarkata agar mesin (XLM maupun TF-IDF) bisa memotong kata dengan presisi.

In [15]:
# 1. Definisikan pola untuk mendeteksi mention (@username)
pola_mention = r"@\S+"

# 2. Fungsi untuk mengecek mention (aman dari data kosong)
def cek_mention(data):
    return data["comment"].astype(str).str.contains(pola_mention, regex=True)

# 3. Tampilkan jumlah komentar yang ada mention-nya
print(f"Komentar yang mengandung mention: {cek_mention(df).sum()}")

# 4. Tampilkan 5 contoh komentarnya di dalam tabel rapi
print("\nContoh 5 komentar yang ada mention-nya:")
df_contoh_mention = pd.DataFrame({
    "Komentar": df.loc[cek_mention(df), "comment"].head(5).str[:150].str.replace("\n", " ")
})

# Menampilkan tabel
display(df_contoh_mention)

Komentar yang mengandung mention: 923

Contoh 5 komentar yang ada mention-nya:


,Komentar
21,@SepulangSekolah bahas juga soal perusahaan AI...
22,@SepulangSekolah gas bang koi
23,@SepulangSekolah y
26,@albertcraftid2584 jokes SMP 😂😂😂😂😂😂😂
40,@FahrezaMuhammad-d3j nah tapi jadi perseteruan...


In [16]:
# 1. EKSEKUSI PENGHAPUSAN: Ganti mention (@username) menjadi spasi kosong
df["comment"] = df["comment"].astype(str).str.replace(r"@\S+", " ", regex=True)

# 2. Rapikan spasi ganda/berlebihan akibat penghapusan kata
df["comment"] = df["comment"].str.replace(r"\s+", " ", regex=True).str.strip()

# 3. Print verifikasi hasil akhir
print("✅ Semua mention (@username) berhasil dibersihkan dari dalam komentar!")

✅ Semua mention (@username) berhasil dibersihkan dari dalam komentar!


In [17]:
df['comment'].duplicated().sum()

np.int64(43)